# Weather Data Processing with Polars (2015–2025)

This notebook processes daily weather observations collected from NOAA weather stations in Poland using Polars.

Main objectives:
- load and process prepared weather data efficiently,
- filter observations for Polish weather stations,
- calculate temperature and weather metric aggregates,
- compare data availability across stations and metrics,
- prepare analysis-ready summary tables for further exploration.

In [1]:
# Import libraries
import os  # Used to read environment variables with os.getenv().
from dotenv import load_dotenv  # Used to load variables from the .env file.
from sqlalchemy import create_engine
from pathlib import Path
import polars as pl

## Database configuration for sample analysis

This notebook uses the Docker PostgreSQL database with sample weather data.

The sample database is used to validate the Bronze → Silver → Gold pipeline and to demonstrate how the analysis can read from the Gold layer.

The full NOAA dataset is not loaded into Docker by default.

In [ ]:
# Define project paths and load local database credentials from the .env file
PROJECT_PATH = Path(
    r"C:\Users\zychl\Documents\GitHub\data-toolkit\weather_data_processing"
)

PLOTS_PATH = PROJECT_PATH / "02_plots"

load_dotenv(PROJECT_PATH / ".env") 

# PostgreSQL connection details
username = os.getenv('POSTGRES_USER')
password = os.getenv('POSTGRES_PASSWORD')
host = 'localhost'
port = os.getenv('HOST_POSTGRES_PORT')
database = os.getenv('POSTGRES_DB')

engine = create_engine(
    f'postgresql+pg8000://{username}:{password}@{host}:{port}/{database}'
    )

# Fetch the data from PostgreSQL analytics_db server - gold layer.
# Open a temporary SQLAlchemy connection for this Polars query.
with engine.connect() as conn:
    df_raw = pl.read_database(
        query='SELECT * FROM gold.weather_observations',
        connection=conn
    )

## Database configuration for full analysis

This notebook uses the local full PostgreSQL database for analysis, not the Docker demo database.

The Docker database contains only sample data and is used to validate the Bronze → Silver → Gold pipeline.
The local analysis database contains the full prepared dataset used for the 2015–2025 analysis.

In [ ]:
PROJECT_PATH = Path(
    r"C:\Users\zychl\Documents\GitHub\data-toolkit\weather_data_processing"
)

PLOTS_PATH = PROJECT_PATH / "02_plots"

# Load database configuration for the local full analysis database.
# This is intentionally separate from the Docker .env file, because Docker uses
# sample data for pipeline validation, while this notebook analyzes the local
# full prepared dataset
load_dotenv(PROJECT_PATH / ".env.analysis") 

username = os.getenv('ANALYSIS_DB_USER')
password = os.getenv('ANALYSIS_DB_PASSWORD')
host = 'localhost'
port = os.getenv('ANALYSIS_DB_PORT')
database = os.getenv('ANALYSIS_DB_NAME')

engine = create_engine(
    f'postgresql+pg8000://{username}:{password}@{host}:{port}/{database}'
    )

with engine.connect() as conn:
    df_raw = pl.read_database(
        query='SELECT * FROM gold.weather_observations',
        connection=conn
    )

### Inspect df

In [ ]:
# Check min and max dates
df_raw.select([
    pl.col("observation_date").min().alias("min_date"),
    pl.col("observation_date").max().alias("max_date")
])

min_date,max_date
date,date
2015-01-01,2026-06-29


In [ ]:
# Preview the DataFrame
df_raw.head()

station,station_name,elevation,observation_date,month_year,metric,value
str,str,"decimal[38,1]",date,str,str,"decimal[38,1]"
"""PL000012120""","""Leba""",2.0,2015-01-01,"""01-2015""","""TMAX""",5.4
"""PL000012120""","""Leba""",2.0,2015-01-01,"""01-2015""","""TMIN""",1.2
"""PL000012120""","""Leba""",2.0,2015-01-01,"""01-2015""","""PRCP""",0.0
"""PL000012120""","""Leba""",2.0,2015-01-01,"""01-2015""","""TAVG""",4.0
"""PL000012385""","""Siedlce""",152.0,2015-01-01,"""01-2015""","""TMAX""",1.5


In [ ]:
# Check DataFrame shape
df_raw.shape

(137418, 7)

In [ ]:
# Check column names and data types
df_raw.schema

Schema([('station', String),
        ('station_name', String),
        ('elevation', Decimal(precision=38, scale=1)),
        ('observation_date', Date),
        ('month_year', String),
        ('metric', String),
        ('value', Decimal(precision=38, scale=1))])

In [ ]:
# Check memory used by DataFrame
df_raw.estimated_size("mb")

8.647311210632324

In [ ]:
# Count null values in each column
df_raw.null_count()

station,station_name,elevation,observation_date,month_year,metric,value
u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0


In [10]:
# Count duplicated rows
df_raw.is_duplicated().sum()

0

In [ ]:
# Count unique values in each column
df_raw.select(pl.all().n_unique())

station,station_name,elevation,observation_date,month_year,metric,value
u32,u32,u32,u32,u32,u32,u32
10,10,10,4175,138,5,785


In [12]:
df_raw.select([
    pl.col('station_name').unique().alias('unique_stations')
])


unique_stations
str
"""Siedlce"""
"""Szczecin"""
"""Wlodawa"""
"""Okecie"""
"""Elblag-Milejewo"""
"""Bialystok"""
"""Leba"""
"""Lawica"""
"""Balice"""
